# Bank ATM Cash Transportation Network Optimization
## Shortest Route / Shortest Path Problem — MIS Network Optimization

**Problem:** A bank needs to transport cash from the General Vault to ATMs distributed across the city. The goal is to find the lowest-cost route to each ATM.

**Model:** Shortest Route / Shortest Path Problem  
**Algorithm:** Dijkstra's Algorithm  
**Tools:** Python + NetworkX

## 1. Import Libraries

In [ ]:
import sys
sys.path.insert(0, '../src')

import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import pandas as pd

print('Libraries loaded successfully.')
print(f'NetworkX version: {nx.__version__}')

## 2. Load and Inspect the Data

In [ ]:
df = pd.read_csv('../data/network_data.csv')
print(f'Total number of routes (edges): {len(df)}')
df

## 3. Build the Graph

A directed weighted graph is constructed. Edge weight = transportation cost (TL per trip).
Nodes are categorized as: `main_depot`, `regional_center`, or `atm`.

In [ ]:
G = nx.DiGraph()

node_types = {
    'General_Vault':   'main_depot',
    'Levent_Center':   'regional_center',
    'Umraniye_Center': 'regional_center',
    'ATM_Besiktas':    'atm',
    'ATM_Sisli':       'atm',
    'ATM_Maslak':      'atm',
    'ATM_Kagithane':   'atm',
    'ATM_Bayrampasa':  'atm',
    'ATM_Kadikoy':     'atm',
    'ATM_Uskudar':     'atm',
    'ATM_Maltepe':     'atm',
}

for node, ntype in node_types.items():
    G.add_node(node, node_type=ntype)

for _, row in df.iterrows():
    G.add_edge(row['source'], row['target'],
               weight=row['cost'],
               distance_km=row['distance_km'])

print(f'Number of nodes : {G.number_of_nodes()}')
print(f'Number of edges : {G.number_of_edges()}')
print(f'Node list       : {list(G.nodes())}')

## 4. Shortest Path Analysis (Dijkstra's Algorithm)

For each ATM, the minimum-cost route starting from the General Vault is computed using Dijkstra's algorithm.

In [ ]:
atm_nodes = [n for n, d in G.nodes(data=True) if d.get('node_type') == 'atm']
results = {}

for atm in atm_nodes:
    path = nx.dijkstra_path(G, 'General_Vault', atm, weight='weight')
    cost = nx.dijkstra_path_length(G, 'General_Vault', atm, weight='weight')
    total_dist = sum(G[path[i]][path[i+1]]['distance_km'] for i in range(len(path)-1))
    results[atm] = {'path': path, 'cost_tl': cost, 'distance_km': total_dist}

# Build results table
rows = []
for atm, data in results.items():
    rows.append({
        'ATM':              atm.replace('ATM_', ''),
        'Optimal Route':    ' -> '.join(data['path']),
        'Total Cost (TL)':  data['cost_tl'],
        'Distance (km)':    data['distance_km']
    })

df_res = pd.DataFrame(rows).sort_values('Total Cost (TL)').reset_index(drop=True)
df_res

## 5. Cost Comparison: Optimal vs. Direct Routes

For ATMs that have a direct connection from the General Vault, we compare the direct cost against the optimal (via Regional Center) cost.

In [ ]:
direct_costs = {
    'ATM_Besiktas': 200,
    'ATM_Sisli':    190,
    'ATM_Kadikoy':  230,
    'ATM_Maltepe':  280,
}

print('Cost Comparison — Direct Route vs Optimal Route:')
print('-' * 60)
for atm, direct in direct_costs.items():
    optimal = results[atm]['cost_tl']
    savings = direct - optimal
    pct = (savings / direct) * 100
    print(f'{atm:20s}:  Direct={direct} TL   Optimal={optimal} TL   Savings={savings} TL ({pct:.0f}%)')

## 6. Network Visualization

In [ ]:
from solution import visualize_network
visualize_network(G, results, node_types, '../results/network_visualization.png')

from IPython.display import Image
Image('../results/network_visualization.png')

## 7. Managerial Interpretation

**Key Findings:**
- Routing through Regional Cash Centers is more cost-effective than direct routes for all ATMs.
- **Levent Center** serves the European Side ATMs: Maslak, Sisli, Kagithane, Besiktas, Bayrampasa (5 ATMs).
- **Umraniye Center** serves the Anatolian Side ATMs: Uskudar, Kadikoy, Maltepe (3 ATMs).
- Total optimal daily transportation cost: **1,520 TL** (assuming one trip to each ATM per day).

**Strategic Recommendation:**  
Bayrampasa ATM is the most expensive stop on the European side (200 TL). If demand in that area grows, establishing a new Regional Cash Center nearby would reduce costs significantly.